## Scenario 1

### Part 1: Data Understanding & Exploration

In [ ]:
import pandas as pd
import numpy as np

# Load Excel file
sales_df = pd.read_excel("sales_data_raw.xlsx", sheet_name="Sales")
targets_df = pd.read_excel("sales_data_raw.xlsx", sheet_name="Targets")

# Preview data
sales_df.head()


In [ ]:
# Dataset overview
sales_df.shape


In [ ]:
sales_df.columns


In [ ]:
sales_df.dtypes

In [ ]:
# Missing values count
sales_df.isnull().sum()


In [ ]:
# Unique values
print(sales_df["Region"].unique())
print(sales_df["Product"].unique())

### Part 2: Data Cleaning & Preparation

In [ ]:
# Fill missing Quantity with 1
sales_df["Quantity"] = sales_df["Quantity"].fillna(1)


In [ ]:
# Recalculate Revenue where missing
sales_df["Revenue"] = sales_df["Revenue"].fillna(
    sales_df["Quantity"] * sales_df["Unit_Price"]
)

In [ ]:
# Missing values count
sales_df.isnull().sum()


In [ ]:
print(sales_df['Order_ID'].nunique())
print(sales_df.shape)

In [ ]:
# Identify duplicates
sales_df[sales_df.duplicated(subset="Order_ID", keep=False)]


In [ ]:
# Remove duplicate Order_IDs (keep first)
sales_df = sales_df.drop_duplicates(subset="Order_ID", keep="first")

print(sales_df['Order_ID'].nunique())
print(sales_df.shape)

In [ ]:
# Standardize Region names
sales_df["Region"] = (
    sales_df["Region"]
    .str.strip()
    .str.title() # .str.title() capitalizes first letter of each word
)

In [ ]:
# Clean email column
sales_df["Customer_Email"] = (
    sales_df["Customer_Email"]
    .str.replace(" ", "", regex=False)
    .str.lower()
)


In [ ]:
# Convert Order_Date to datetime
sales_df["Order_Date"] = pd.to_datetime(sales_df["Order_Date"])


In [ ]:
# Ensure numeric columns
sales_df["Quantity"] = sales_df["Quantity"].astype(int)
sales_df["Unit_Price"] = sales_df["Unit_Price"].astype(float)
sales_df["Revenue"] = sales_df["Revenue"].astype(float)


### Part 3: Transformation & Aggregation

In [ ]:
north_sales = sales_df[sales_df["Region"] == "North"]


In [ ]:
north_sales_sorted = north_sales.sort_values(by="Revenue", ascending=False)
north_sales_sorted.head()


In [ ]:
# Revenue by region
revenue_by_region = (
    sales_df
    .groupby("Region")["Revenue"]
    .sum()
    .reset_index()
)
print(revenue_by_region)


In [ ]:
# Top product per region
top_product_region = (
    sales_df
    .groupby(["Region", "Product"])["Revenue"]
    .sum()
    .reset_index()
    .sort_values(["Region", "Revenue"], ascending=[True, False])
    .groupby("Region")
    .head(1)
)
print(top_product_region)

In [ ]:
# Total revenue by sales rep
rep_performance = (
    sales_df
    .groupby("Sales_Rep")["Revenue"]
    .sum()
    .reset_index()
)
print(rep_performance)

In [ ]:
# Rank sales reps
rep_performance["Rank"] = rep_performance["Revenue"].rank(
    ascending=False, method="dense"
)

rep_performance.sort_values("Rank")


#### Merging Targets & KPI Creation

In [ ]:
# Merge revenue summary with targets
region_kpi = revenue_by_region.merge(
    targets_df,
    on="Region",
    how="left"
)

print(region_kpi)

In [ ]:
# Achievement percentage
region_kpi["Achievement_%"] = (
    region_kpi["Revenue"] / region_kpi["Monthly_Target"] * 100
)
region_kpi

In [ ]:
# Status classification
region_kpi["Status"] = np.where(
    region_kpi["Achievement_%"] >= 100,
    "Achieved",
    "Not Achieved"
)

region_kpi


In [ ]:
sales_df.to_excel("sales_cleaned.xlsx", index=False)
